## Scenario 1: A single data scientist participating in an ML competition

This scenario demonstrates how an individual data scientist can use MLflow to track machine learning experiments on their local machine. This is a common setup for solo projects, hackathons, or competitions where collaboration and remote access are not required.

### MLflow setup overview
- **Tracking server:** Not used (runs locally, no remote server)
- **Backend store:** Local filesystem (stores experiment metadata in the `mlruns/` folder)
- **Artifacts store:** Local filesystem (stores model files and other artifacts in the same `mlruns/` folder)

With this setup, all experiment runs, parameters, metrics, and artifacts are saved locally. You can explore and compare your experiments using the MLflow UI.

### How to use the MLflow UI
- You can launch the MLflow UI by running `mlflow ui` in your terminal, or by running the provided code cell in this notebook.
- The UI will be available at [http://localhost:5000](http://localhost:5000).
- Use the UI to browse experiments, compare runs, and inspect logged models and artifacts.

> **Tip:** This local setup is ideal for learning and prototyping. For team projects or production, you would typically use a remote tracking server and a more robust backend (e.g., a database and cloud storage).

In [1]:
%pip install mlflow
import mlflow


[notice] A new release of pip is available: 24.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


/Users/sebastiaoclemente/Documents/Business Analytics Masters IE/TERM 2/MLOPS/ie-mlops-nys-taxis/venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
print(f"tracking URI: '{mlflow.get_tracking_uri()}'")

tracking URI: 'sqlite:////Users/sebastiaoclemente/Documents/Business%20Analytics%20Masters%20IE/TERM%202/MLOPS/ie-mlops-nys-taxis/mlflow.db'


In [3]:
mlflow.search_experiments()

2026/03/08 16:06:28 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/08 16:06:28 INFO mlflow.store.db.utils: Updating database tables


[<Experiment: artifact_location=('/Users/sebastiaoclemente/Documents/Business Analytics Masters IE/TERM '
  '2/MLOPS/ie-mlops-nys-taxis/mlruns/0'), creation_time=1772982388609, experiment_id='0', last_update_time=1772982388609, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

### Creating an experiment and logging a new run

In [4]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np
import mlflow
import sklearn
import datetime
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

mlflow.set_experiment("my-experiment-1")

X, y = load_iris(return_X_y=True)
class_names = load_iris().target_names

models = [
    ("LogisticRegression", LogisticRegression(C=1.0, random_state=42, max_iter=1000, solver="lbfgs")),
    ("RandomForestClassifier", RandomForestClassifier(n_estimators=100, random_state=42))
]

for model_name, model in models:
    with mlflow.start_run() as run:
        mlflow.set_tag("model_type", model_name)
        mlflow.log_param("sklearn_version", sklearn.__version__)
        model.fit(X, y)
        y_pred = model.predict(X)
        acc = accuracy_score(y, y_pred)
        mlflow.log_metric("accuracy", acc)
        # Log confusion matrix as labeled DataFrame
        cm = confusion_matrix(y, y_pred)
        cm_df = pd.DataFrame(cm, index=class_names, columns=class_names)
        cm_df.to_csv("confusion_matrix_labeled.csv")
        mlflow.log_artifact("confusion_matrix_labeled.csv")
        # Confusion matrix heatmap as image
        plt.figure(figsize=(5,4))
        sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues")
        plt.title(f"Confusion Matrix ({model_name})")
        plt.ylabel("True label")
        plt.xlabel("Predicted label")
        plt.tight_layout()
        plt.savefig("confusion_matrix_heatmap.png")
        plt.close()
        mlflow.log_artifact("confusion_matrix_heatmap.png")
        # Provide input_example and use 'name' instead of deprecated 'artifact_path'
        input_example = np.expand_dims(X[0], axis=0)
        mlflow.sklearn.log_model(model, name="models", input_example=input_example)
        mlflow.set_tag("n_classes", len(np.unique(y)))
        mlflow.set_tag("run_time", datetime.datetime.now().isoformat())
        mlflow.set_tag("description", f"{model_name} on Iris dataset")
        print(f"Logged run for {model_name}, accuracy={acc:.3f}")

2026/03/08 16:07:08 INFO mlflow.tracking.fluent: Experiment with name 'my-experiment-1' does not exist. Creating a new experiment.
2026/03/08 16:07:08 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for LogisticRegression, accuracy=0.973


2026/03/08 16:07:11 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged run for RandomForestClassifier, accuracy=1.000


In [5]:
mlflow.search_experiments()

[<Experiment: artifact_location=('/Users/sebastiaoclemente/Documents/Business Analytics Masters IE/TERM '
  '2/MLOPS/ie-mlops-nys-taxis/mlruns/1'), creation_time=1772982428062, experiment_id='1', last_update_time=1772982428062, lifecycle_stage='active', name='my-experiment-1', tags={}, workspace='default'>,
 <Experiment: artifact_location=('/Users/sebastiaoclemente/Documents/Business Analytics Masters IE/TERM '
  '2/MLOPS/ie-mlops-nys-taxis/mlruns/0'), creation_time=1772982388609, experiment_id='0', last_update_time=1772982388609, lifecycle_stage='active', name='Default', tags={}, workspace='default'>]

In [6]:
# Launch MLflow UI (default: uses local mlruns/ folder)
import subprocess
import sys

# This will run 'mlflow ui' as a background process
subprocess.Popen([sys.executable, '-m', 'mlflow', 'ui'])
print("MLflow UI started. Open http://localhost:5000 in your browser.")

MLflow UI started. Open http://localhost:5000 in your browser.


Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.


2026/03/08 16:07:23 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/03/08 16:07:23 INFO mlflow.store.db.utils: Updating database tables
[MLflow] Security middleware enabled with default settings (localhost-only). To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
INFO:     Uvicorn running on http://127.0.0.1:5000 (Press CTRL+C to quit)
INFO:     Started parent process [73207]
2026/03/08 16:07:24 INFO mlflow.server.jobs.utils: Starting huey consumer for job function invoke_scorer
2026/03/08 16:07:24 INFO mlflow.server.jobs.utils: Starting huey consumer for job function optimize_prompts
2026/03/08 16:07:24 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_session_scorer
2026/03/08 16:07:24 INFO mlflow.server.jobs.utils: Starting huey consumer for job function run_online_trace_scorer
2026/03/08 16:07:24 INFO mlflow.server.jobs.utils: Starting dedicated Huey consumer

INFO:     127.0.0.1:50470 - "GET / HTTP/1.1" 200 OK
INFO:     127.0.0.1:50470 - "GET /static-files/static/js/main.c10baf8d.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:50471 - "GET /static-files/static/css/main.e1790ccd.css HTTP/1.1" 200 OK
INFO:     127.0.0.1:50471 - "GET /ajax-api/3.0/mlflow/server-info HTTP/1.1" 200 OK
INFO:     127.0.0.1:50471 - "GET /ajax-api/3.0/mlflow/assistant/config HTTP/1.1" 200 OK
INFO:     127.0.0.1:50471 - "GET /static-files/static/js/1521.91cd4ff9.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:50470 - "GET /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK
INFO:     127.0.0.1:50470 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=5&order_by=last_update_time+DESC HTTP/1.1" 200 OK
INFO:     127.0.0.1:50470 - "GET /static-files/static/js/4783.3620eca9.chunk.js HTTP/1.1" 200 OK
INFO:     127.0.0.1:50470 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC&filter=tags.%60mlflow.experiment.isGateway%60+IS+NULL HTTP/1.

### Interacting with the model registry

In [7]:
from mlflow.tracking import MlflowClient


client = MlflowClient()

In [ ]:
from mlflow.exceptions import MlflowException

try:
    client.search_registered_models()
except MlflowException:
    print("It's not possible to access the model registry :(")

INFO:     127.0.0.1:50480 - "POST /ajax-api/3.0/mlflow/ui-telemetry HTTP/1.1" 200 OK


2026/03/08 16:08:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2.
[2026-03-08 16:08:26,132] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2.
2026/03/08 16:08:32 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2
[2026-03-08 16:08:32,728] INFO:huey:Worker-2:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2
2026/03/08 16:08:32 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2 executed in 0.009s
[2026-03-08 16:08:32,738] INFO:huey:Worker-2:mlflow.server.jobs.utils.online_scoring_scheduler: 325f5fc2-8ea8-487c-994c-d7697cc40ae2 executed in 0.009s


INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/experiments/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "GET /api/2.0/mlflow/experiments/get-by-name?experiment_name=my-experiment-1 HTTP/1.1" 404 Not Found
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/experiments/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "GET /api/2.0/mlflow/experiments/get?experiment_id=1 HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-batch HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-parameter HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-metric HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-parameter HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-parameter HTTP/1.1" 200 OK
INFO:     127.0.0.1:50499 - "POST /api/2.0/mlflow/runs/log-parameter HTTP/1.1" 200 OK
INFO:     127

2026/03/08 16:09:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 7c22e646-c75d-4ffe-8c72-e0f2f604db9a.
[2026-03-08 16:09:26,135] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 7c22e646-c75d-4ffe-8c72-e0f2f604db9a.


INFO:     127.0.0.1:50515 - "GET /api/2.0/mlflow/registered-models/search?filter=tag.%60mlflow.prompt.is_prompt%60+%21%3D+%27true%27&max_results=100 HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "POST /api/2.0/mlflow/runs/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "POST /api/2.0/mlflow/registered-models/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "GET /api/2.0/mlflow/runs/get?run_uuid=66c38f266b314787a8b9a58641da3e0b&run_id=66c38f266b314787a8b9a58641da3e0b HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "GET /api/2.0/mlflow-artifacts/artifacts?path=1%2F66c38f266b314787a8b9a58641da3e0b%2Fartifacts%2Fmodels HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "GET /api/2.0/mlflow/runs/get?run_uuid=66c38f266b314787a8b9a58641da3e0b&run_id=66c38f266b314787a8b9a58641da3e0b HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "POST /api/2.0/mlflow/logged-models/search HTTP/1.1" 200 OK
INFO:     127.0.0.1:50515 - "POST /api/2.0/mlflow/model-versions/create HTTP/1.1" 200 OK
INFO:     127.0.0.1:505

...

2026/03/08 16:55:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f.
[2026-03-08 16:55:26,144] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f.
2026/03/08 16:55:27 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f
[2026-03-08 16:55:27,618] INFO:huey:Worker-2:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f
2026/03/08 16:55:27 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f executed in 0.013s
[2026-03-08 16:55:27,632] INFO:huey:Worker-2:mlflow.server.jobs.utils.online_scoring_scheduler: 824e938a-e786-494b-af49-c0704a38259f executed in 0.013s
2026/03/08 16:56:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.u

INFO:     127.0.0.1:50960 - "GET /ajax-api/2.0/mlflow/experiments/search?max_results=25&order_by=last_update_time+DESC&filter=tags.%60mlflow.experiment.isGateway%60+IS+NULL HTTP/1.1" 200 OK


2026/03/08 17:00:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2.
[2026-03-08 17:00:26,137] INFO:huey.consumer.Scheduler:Scheduler:Enqueueing periodic task mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2.
2026/03/08 17:00:34 INFO huey: Executing mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2
[2026-03-08 17:00:34,238] INFO:huey:Worker-1:Executing mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2
2026/03/08 17:00:34 INFO huey: mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2 executed in 0.003s
[2026-03-08 17:00:34,241] INFO:huey:Worker-1:mlflow.server.jobs.utils.online_scoring_scheduler: 32bda95d-0a90-4679-bfd3-a8cdcaec7aa2 executed in 0.003s
2026/03/08 17:01:26 INFO huey.consumer.Scheduler: Enqueueing periodic task mlflow.server.jobs.u

INFO:     127.0.0.1:50991 - "GET /api/2.0/mlflow/runs/get?run_uuid=f9ec9524c3c94b0f9800c7e84f470443&run_id=f9ec9524c3c94b0f9800c7e84f470443 HTTP/1.1" 200 OK
INFO:     127.0.0.1:50991 - "GET /api/2.0/mlflow-artifacts/artifacts?path=2%2Ff9ec9524c3c94b0f9800c7e84f470443%2Fartifacts%2Fmodel HTTP/1.1" 200 OK
INFO:     127.0.0.1:50991 - "GET /api/2.0/mlflow-artifacts/artifacts/2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model HTTP/1.1" 500 Internal Server Error
INFO:     127.0.0.1:50991 - "GET /api/2.0/mlflow-artifacts/artifacts/2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model HTTP/1.1" 500 Internal Server Error


2026/03/08 17:01:33 ERROR mlflow.server.handlers: Error in _download_artifact: The following failures occurred while downloading one or more artifacts from ./mlartifacts:
##### File 2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model #####
[Errno 2] No such file or directory: './mlartifacts/2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model'. Set MLFLOW_LOGGING_LEVEL=DEBUG for traceback.
2026/03/08 17:01:33 ERROR mlflow.server.handlers: Error in _download_artifact: The following failures occurred while downloading one or more artifacts from ./mlartifacts:
##### File 2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model #####
[Errno 2] No such file or directory: './mlartifacts/2/f9ec9524c3c94b0f9800c7e84f470443/artifacts/model'. Set MLFLOW_LOGGING_LEVEL=DEBUG for traceback.
